In [1]:
%pip install -q 'langchain>=1,<2' 'langchain-groq<2'

In [2]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Prompt for key if not found, or if you want to update it
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API key: ").strip()

print("Groq key loaded:", bool(os.environ.get("GROQ_API_KEY")))

Groq API key: ··········
Groq key loaded: True


In [3]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
import os

# Initialize model using the loaded API key
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    timeout=30,
    max_retries=1,
    groq_api_key=os.environ.get("GROQ_API_KEY")
)

In [4]:
@tool
def add(a: int, b: int) -> int:
    """Add two whole-number quantities when they are combined."""
    return a + b

print(add.invoke({"a": 5, "b": 10}))  # expected: 15

15


In [5]:
from math import isfinite, radians, tan

@tool
def tree_distance(height_m: float, angle_deg: float) -> float | str:
    """Find horizontal tree distance on level ground from height above eye level in metres and angle of elevation in degrees."""
    if not isfinite(height_m) or height_m <= 0:
        return "Invalid height. Supply a positive finite height above eye level."
    if not isfinite(angle_deg) or not 0 < angle_deg < 90:
        return "Invalid angle. Supply an angle strictly between 0 and 90 degrees."
    return round(height_m / tan(radians(angle_deg)), 2)

print(tree_distance.invoke({"height_m": 12, "angle_deg": 30}))  # expected: 20.78

20.78


In [6]:
INSTRUCTIONS = (
    "You are the Intelligent Calculator. Use add for combining two whole-number "
    "quantities. Use tree_distance only for horizontal distance on level ground "
    "when height above eye level in metres and angle of elevation in degrees "
    "are supplied. For supported complete problems, call the matching tool once "
    "and use its result. If a required value is missing, ask for it before "
    "calling a tool. Never guess inputs. If a tool reports an invalid input, ask "
    "for a correction. For other requests, explain what you can calculate. "
    "Give the final answer with units."
)
# Recreate agent with the updated model config
agent = create_agent(model=model, tools=[add, tree_distance], system_prompt=INSTRUCTIONS)

In [7]:
def show_run(problem):
    print("QUESTION:", problem, "\n")
    result = agent.invoke(
        {"messages": [("human", problem)]},
        config={"recursion_limit": 8, "run_name": "intro_calculator"},
    )
    for m in result["messages"][1:]:
        if m.type == "ai" and getattr(m, "tool_calls", None):
            for call in m.tool_calls:
                print("TOOL CALL  :", call["name"], call["args"])
        elif m.type == "tool":
            print("TOOL RESULT:", m.content)
        elif m.type == "ai":
            print("\nFINAL ANSWER:", m.content)
    return result

In [8]:
apples_run = show_run("One person brings five apples and another brings ten. How many apples do they have together?")

QUESTION: One person brings five apples and another brings ten. How many apples do they have together? 

TOOL CALL  : add {'a': 5, 'b': 10}
TOOL RESULT: 15

FINAL ANSWER: They have **15 apples** in total.


In [9]:
tree_run = show_run("The top of a tree is 12 metres above my eye level and the angle of elevation is 30 degrees. How far am I from the tree?")

QUESTION: The top of a tree is 12 metres above my eye level and the angle of elevation is 30 degrees. How far am I from the tree? 

TOOL CALL  : tree_distance {'angle_deg': 30, 'height_m': 12}
TOOL RESULT: 20.78

FINAL ANSWER: The horizontal distance from you to the tree is **20.78 m**.


In [10]:
---

QUESTION: One person brings seven apples and another brings eight. How many apples do they have together? 

TOOL CALL  : add {'a': 7, 'b': 8}
TOOL RESULT: 15

FINAL ANSWER: The total number of apples is **15 apples**.


---

In [11]:
missing_angle_run = show_run("How far away is a tree that is 15 metres above my eye level?")

QUESTION: How far away is a tree that is 15 metres above my eye level? 


FINAL ANSWER: To determine the horizontal distance to the tree, I need both:

* the height above your eye level (you’ve given 15 metres), **and**
* the angle of elevation from your eye to the top of the tree (in degrees).

Could you tell me the angle of elevation?


In [12]:
fresh_run = show_run("One person brings seven apples and another brings eight. How many apples do they have together?")

QUESTION: One person brings seven apples and another brings eight. How many apples do they have together? 

TOOL CALL  : add {'a': 7, 'b': 8}
TOOL RESULT: 15

FINAL ANSWER: The total number of apples is **15 apples**.


In [18]:
%pip install mcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mcp]


In [21]:
import sys
import os
import subprocess

def clean_and_install_mcp():
    print("Enforcing mcp<2 version and clearing module caches...")
    # Downgrade packages cleanly
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "mcp", "mcp-types"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mcp<2"])

    # Purge any imported mcp module cache from memory
    for key in list(sys.modules.keys()):
        if key == "mcp" or key.startswith("mcp."):
            del sys.modules[key]

try:
    from mcp.server.fastmcp import FastMCP
    from mcp.types import ToolAnnotations
except (ImportError, ModuleNotFoundError):
    clean_and_install_mcp()
    from mcp.server.fastmcp import FastMCP
    from mcp.types import ToolAnnotations

mcp = FastMCP(
    "Intelligent Calculator",
    host=os.environ.get("INTRO_MCP_HOST", "127.0.0.1"),
    stateless_http=True,
    json_response=True,
)

@mcp.tool(annotations=ToolAnnotations(readOnlyHint=True, openWorldHint=False))
def solve_math_problem(problem: str) -> str:
    """Solve an apples addition or tree-distance word problem. Pass the original wording."""
    problem = problem.strip()
    if not 1 <= len(problem) <= 1500:
        return "Enter an English word problem of 1 to 1500 characters."
    result = agent.invoke(
        {"messages": [("human", problem)]},
        config={"recursion_limit": 8, "run_name": "intro_calculator_mcp"},
    )
    return result["messages"][-1].content

print("MCP tool registered: solve_math_problem")

Enforcing mcp<2 version and clearing module caches...
MCP tool registered: solve_math_problem


In [22]:
print("python server.py            # terminal 1, listens on http://127.0.0.1:8000/mcp")
print("tunnel-client run --profile calculator   # terminal 2, keep it running")

python server.py            # terminal 1, listens on http://127.0.0.1:8000/mcp
tunnel-client run --profile calculator   # terminal 2, keep it running
